# LAPQUE RVC v2 — HUẤN LUYỆN VOICE CLONING (LỘC ĐỈNH KÝ)
Huấn luyện mạng nơ-ron nhân bản giọng nói thực sự trên Google Colab Free T4 GPU.

In [ ]:
# BƯỚC 1: Cài đặt thư viện RVC v2 và tải Pretrained Base Weights
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC
!pip install -r requirements.txt
!pip install pycloudflared soundfile uvicorn fastapi httpx python-multipart

!mkdir -p assets/hubert assets/rmvpe assets/pretrained_v2
!wget -O assets/hubert/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!wget -O assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
!wget -O assets/pretrained_v2/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth
!wget -O assets/pretrained_v2/f0D40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth
print("-> Đã tải xong Base Models!")

In [ ]:
# BƯỚC 2: Giải nén 300 mẫu audio Lộc Đỉnh Ký
import os
import zipfile

dataset_dir = "/content/RVC/dataset_loc_dinh_ky"
os.makedirs(dataset_dir, exist_ok=True)

if os.path.exists("/content/dataset_loc_dinh_ky_40k.zip"):
    with zipfile.ZipFile("/content/dataset_loc_dinh_ky_40k.zip", 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"-> Đã giải nén thành công {len(os.listdir(dataset_dir))} file audio Lộc Đỉnh Ký!")
else:
    print("Vui lòng kéo thả file dataset_loc_dinh_ky_40k.zip vào thư mục /content bên trái!")

In [ ]:
# BƯỚC 3: Trích xuất đặc trưng F0 (RMVPE) & HuBERT Semantic Features
%cd /content/RVC
!python infer/modules/train/preprocess.py /content/RVC/dataset_loc_dinh_ky 40000 2 /content/RVC/logs/loc_dinh_ky False 3.0
!python infer/modules/train/extract/extract_f0_rmvpe.py 2 0 0 /content/RVC/logs/loc_dinh_ky True
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 0 /content/RVC/logs/loc_dinh_ky v2
print("-> Trích xuất đặc trưng giọng nói hoàn tất!")

In [ ]:
# BƯỚC 4: Huấn luyện mạng nơ-ron 150 Epochs (~15 phút)
%cd /content/RVC
!python infer/modules/train/train.py -e loc_dinh_ky -sr 40k -f0 1 -bs 8 -g 0 -te 150 -se 25 -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 1 -c 0 -sw 1 -v v2
!python infer/modules/train/train_index.py /content/RVC/logs/loc_dinh_ky v2
print("-> HUẤN LUYỆN THÀNH CÔNG 100%! ĐÃ TẠO WEIGHTS .PTH VÀ INDEX .INDEX")

In [ ]:
# BƯỚC 5: Khởi động Live API Server & Cloudflare Tunnel
import os
import sys

# Đặt đường dẫn làm việc về /content/RVC trước tiên
os.chdir("/content/RVC")
if "/content/RVC" not in sys.path:
    sys.path.insert(0, "/content/RVC")

import torch
import soundfile as sf
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
import uvicorn
from pycloudflared import try_cloudflare
import threading
import time

app = FastAPI(title="LAPQUE RVC Server")

from infer.modules.vc.modules import VC
from configs.config import Config

config = Config()
config.device = "cuda:0"
config.is_half = True
vc = VC(config)

vc.get_vc("loc_dinh_ky.pth")
index_path = "/content/RVC/logs/loc_dinh_ky/added_IVF256_Flat_Fast_loc_dinh_ky_v2.index"

@app.post("/convert")
async def convert_voice(file: UploadFile = File(...), pitch_shift: int = Form(default=0), index_rate: float = Form(default=0.75)):
    try:
        content = await file.read()
        in_path = "/content/temp_in.wav"
        out_path = "/content/temp_out.wav"
        with open(in_path, "wb") as f:
            f.write(content)

        info, opt = vc.vc_single(
            sid=0,
            input_audio_path=in_path,
            f0_up_key=pitch_shift,
            f0_file=None,
            f0_method="rmvpe",
            file_index=index_path if os.path.exists(index_path) else "",
            file_index2="",
            index_rate=index_rate,
            filter_radius=3,
            resample_sr=0,
            rms_mix_rate=0.25,
            protect=0.33
        )
        if opt is not None:
            sr, wav_data = opt
            sf.write(out_path, wav_data, sr)
            with open(out_path, "rb") as f_res:
                return Response(content=f_res.read(), media_type="audio/wav")
        else:
            raise RuntimeError(str(info))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

t = threading.Thread(target=run_app)
t.daemon = True
t.start()

time.sleep(3)
tunnel = try_cloudflare(port=8000)
print("="*60)
print(f"URL KẾT NỐI VÀO LAPQUE STUDIO:")
print(f" -> {tunnel.tunnel_url}")
print("="*60)
print("Dán URL này vào ô 'Kết nối Google Colab RVC GPU' trong Web Studio!")